# Лабораторная работа №3: RAG-система для работы с PDF

Документ: «Training language models to follow instructions with human feedback» (InstructGPT), arXiv:2203.02155, 68 страниц.

- LLM: `qwen2.5:3b` (Ollama), температура 0
- Эмбеддинги: `bge-m3` (`OllamaEmbeddings`)
- Разбиение: `RecursiveCharacterTextSplitter`, `chunk_size=1000`, `chunk_overlap=200`
- Поиск: косинусное сходство, `k=3`

Запуск на Kaggle: включить Internet и GPU T4, выполнить все ячейки. Первая ячейка ставит Ollama, качает модели и PDF с arXiv. Полный прогон — 20–30 минут. Результаты экспериментов сохраняются в `results/*.json`; с `SKIP_EXPERIMENTS=1` ноутбук читает их и перестраивает только таблицы.

In [1]:
# ============================================================
# ПОДГОТОВКА СРЕДЫ (Kaggle): Ollama, модели, пакеты, PDF
# При локальном запуске с уже установленной Ollama ячейку можно пропустить.
# ============================================================
import os, subprocess, time

# 1. PDF-документ (если не загружен как Kaggle Dataset — скачиваем с arXiv)
if not os.path.exists("LLM_instruct.pdf"):
    !curl -sL -o LLM_instruct.pdf "https://arxiv.org/pdf/2203.02155"
print("PDF на месте:", os.path.exists("LLM_instruct.pdf"))

# 2. Пакеты Python
!pip install -q langchain langchain-core langchain-community langchain-ollama langchain-text-splitters pypdf numpy

# 3. Ollama: установка бинарника и запуск сервера в фоне
# (установщику нужен zstd, в образе Kaggle его нет — ставим через apt)
if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
    !apt-get update -qq && apt-get install -y -qq zstd
    !curl -fsSL https://ollama.com/install.sh | sh

ollama_proc = subprocess.Popen(["ollama", "serve"],
                               stdout=open("/tmp/ollama.log", "w"),
                               stderr=subprocess.STDOUT)
time.sleep(5)

# 4. Модели: генерация + эмбеддинги
!ollama pull qwen2.5:3b
!ollama pull bge-m3
!ollama list

PDF на месте: True
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 130198 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%##############################                                  55.9%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama"

In [2]:
import os, re, json, time, platform
from itertools import combinations
from pathlib import Path

import numpy as np

import langchain, langchain_core, langchain_community, pypdf
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.vectorstores import InMemoryVectorStore

PDF_PATH = "LLM_instruct.pdf"
GEN_MODEL = "qwen2.5:3b"   # генеративная модель (Ollama)
EMB_MODEL = "bge-m3"       # модель эмбеддингов (Ollama), мультиязычная
K = 3                      # top-k для retriever
BASE_CHUNK_SIZE = 1000
BASE_CHUNK_OVERLAP = 200

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
SKIP_EXPERIMENTS = os.environ.get("SKIP_EXPERIMENTS") == "1"

print("Python:", platform.python_version())
print("langchain:", langchain.__version__)
print("langchain-core:", langchain_core.__version__)
print("langchain-community:", langchain_community.__version__)
print("pypdf:", pypdf.__version__)

# Отдельные объекты для генерации и эмбеддингов, создаются до первого вызова
llm = ChatOllama(model=GEN_MODEL, temperature=0, num_predict=350)
emb = OllamaEmbeddings(model=EMB_MODEL)
print("LLM:", GEN_MODEL, "| Embeddings:", EMB_MODEL)
print("SKIP_EXPERIMENTS =", SKIP_EXPERIMENTS)

/tmp/ipykernel_49/3976483233.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain, langchain_core, langchain_community, pypdf


Python: 3.13.16
langchain: 1.4.7
langchain-core: 1.7.0
langchain-community: 0.4.2
pypdf: 7.1.0
LLM: qwen2.5:3b | Embeddings: bge-m3
SKIP_EXPERIMENTS = False


## Проверка базового конвейера

Загружаем PDF постранично, разбиваем на фрагменты, проверяем число страниц и фрагментов.

In [3]:
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print("Страниц в PDF:", len(pages))
print("Символов на стр. 1:", len(pages[0].page_content))
print("Метаданные страницы:", pages[0].metadata)

def make_chunks(chunk_size, chunk_overlap):
    """Явное разбиение страниц на фрагменты с метаданными."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    docs = splitter.split_documents(pages)
    for i, d in enumerate(docs):
        d.metadata["chunk_id"] = i + 1                  # уникальный id фрагмента
        d.metadata["page_pdf"] = d.metadata["page"] + 1  # порядковая страница PDF (с 1)
    return docs

chunks = make_chunks(BASE_CHUNK_SIZE, BASE_CHUNK_OVERLAP)
print(f"\nФрагментов (chunk_size={BASE_CHUNK_SIZE}, overlap={BASE_CHUNK_OVERLAP}):", len(chunks))
print("\nПервый фрагмент (метаданные):", chunks[0].metadata)
print(chunks[0].page_content[:400])

Страниц в PDF: 68
Символов на стр. 1: 2529
Метаданные страницы: {'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-03-07T01:37:12+00:00', 'author': '', 'keywords': '', 'moddate': '2022-03-07T01:37:12+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'LLM_instruct.pdf', 'total_pages': 68, 'page': 0, 'page_label': '1'}

Фрагментов (chunk_size=1000, overlap=200): 246

Первый фрагмент (метаданные): {'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-03-07T01:37:12+00:00', 'author': '', 'keywords': '', 'moddate': '2022-03-07T01:37:12+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'LLM_instruct.pdf', 'total_pages': 68, 'page': 0, 'page_label': '1', 'chunk_id': 1, 'page_pdf'

In [4]:
# ---------------------------------------------------------------
# Общие функции RAG-конвейера
# ---------------------------------------------------------------

def normalize(s):
    """Нормализация текста для сравнения: лигатуры, кавычки, регистр, пробелы."""
    s = (s.replace("\uFB01", "fi").replace("\uFB02", "fl")
           .replace("\u2019", "'").replace("\u2018", "'")
           .replace("\u201c", '"').replace("\u201d", '"')
           .replace("\u2013", "-").replace("\u2014", "-"))
    return re.sub(r"\s+", " ", s).strip().lower()

def build_index(docs):
    """Новый поисковый индекс по фрагментам (пересоздаётся при смене разбиения)."""
    return InMemoryVectorStore.from_documents(docs, emb)

_index_cache = {}
def get_index(key, docs):
    if key not in _index_cache:
        t0 = time.time()
        _index_cache[key] = build_index(docs)
        print(f"Индекс '{key}': {len(docs)} фрагментов, построен за {time.time()-t0:.1f} с")
    return _index_cache[key]

def format_context(docs):
    """Контекст для LLM: перед каждым фрагментом его id, имя файла и страница."""
    parts = []
    for d in docs:
        header = (f"[Фрагмент {d.metadata['chunk_id']}, "
                  f"{Path(d.metadata['source']).name}, стр. {d.metadata['page_pdf']}]")
        parts.append(header + "\n" + d.page_content.strip())
    return "\n\n".join(parts)

BASE_PROMPT = (
    "Ты — ассистент, который отвечает на вопросы по научной статье. "
    "Ниже приведены фрагменты статьи (контекст).\n\n"
    "Контекст:\n{context}\n\n"
    "Вопрос: {question}\n\n"
    "Дай краткий точный ответ на русском языке, опираясь на контекст.\n"
    "Ответ:"
)

STRICT_PROMPT = (
    "Ты — ассистент, который отвечает на вопросы СТРОГО по научной статье. "
    "Ниже приведены фрагменты статьи (контекст).\n\n"
    "Контекст:\n{context}\n\n"
    "Вопрос: {question}\n\n"
    "Правила:\n"
    "1. Используй ТОЛЬКО сведения из контекста выше.\n"
    "2. Если в контексте нет сведений, достаточных для ответа, ответь ровно так: "
    "«В документе нет информации для ответа на этот вопрос».\n"
    "3. Не используй собственные знания и ничего не выдумывай.\n"
    "4. Отвечай кратко, на русском языке.\n"
    "Ответ:"
)

CITATION_PROMPT = (
    "Ты — ассистент, который отвечает на вопросы по научной статье и обязан "
    "ссылаться на источники.\n\n"
    "Ниже приведены фрагменты статьи. Перед каждым фрагментом указаны его "
    "идентификатор, имя файла и номер страницы.\n\n"
    "Контекст:\n{context}\n\n"
    "Вопрос: {question}\n\n"
    "Правила:\n"
    "1. Отвечай кратко, на русском языке, только на основе приведённых фрагментов.\n"
    "2. Каждое фактическое утверждение сопровождай ссылкой вида "
    "[Фрагмент N, стр. M], где N — идентификатор фрагмента, M — номер страницы "
    "из заголовка фрагмента.\n"
    "3. Не придумывай идентификаторы и страницы, используй только переданные фрагменты.\n"
    "Ответ:"
)

def retrieve(store, query, k=K):
    """Retriever вызывается при обработке каждого вопроса."""
    return store.similarity_search(query, k=k)

def ask_llm(prompt):
    return llm.invoke(prompt).content.strip()

def rag(question, store, k=K, prompt_template=BASE_PROMPT, search_query=None, verbose=True):
    """Полный цикл RAG: поиск -> контекст этого поиска -> генерация."""
    query = search_query or question
    retrieved = retrieve(store, query, k)
    context = format_context(retrieved)          # контекст собирается заново
    answer = ask_llm(prompt_template.format(context=context, question=question))
    if verbose:
        print("Вопрос:", question)
        print("Поисковый запрос:", query)
        print("Найденные фрагменты:")
        for d in retrieved:
            preview = re.sub(r"\s+", " ", d.page_content)[:120]
            print(f"  - Фрагмент {d.metadata['chunk_id']}, стр. {d.metadata['page_pdf']}: {preview}...")
        print("Ответ модели:", answer)
        print("-" * 80)
    return {
        "question": question, "query": query, "answer": answer,
        "retrieved": [{"chunk_id": d.metadata["chunk_id"],
                       "page_pdf": d.metadata["page_pdf"],
                       "text": d.page_content} for d in retrieved],
    }

def quote_hit(retrieved, quote):
    """Фрагмент признаётся подходящим, если эталонная цитата целиком входит
    хотя бы в один из найденных фрагментов (в ней достаточно сведений для ответа)."""
    nq = normalize(quote)
    return any(nq in normalize(d["text"]) for d in retrieved)

def page_hit(retrieved, pages):
    """Вспомогательная проверка: найден ли фрагмент с эталонной страницы."""
    return any(d["page_pdf"] in pages for d in retrieved)

def save_results(name, obj):
    with open(RESULTS_DIR / name, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)

def load_results(name):
    with open(RESULTS_DIR / name, encoding="utf-8") as f:
        return json.load(f)

def load_judgments():
    """Ручные оценки (баллы LLM, подтверждение ссылок) заполняются после
    просмотра сырых ответов в results/*.json."""
    p = RESULTS_DIR / "judgments.json"
    return json.load(open(p, encoding="utf-8")) if p.exists() else {}

print("Функции готовы.")

Функции готовы.


In [23]:
# ---------------------------------------------------------------
# РУЧНЫЕ ОЦЕНКИ: заполняются после просмотра сырых ответов
# (баллы LLM по шкале 0-2, подтверждение ссылок, вердикты по отказам).
# Ключи: "t1:<A/B>:<индекс>:<вопрос>", "t5:<номер>", "t6:<промпт>:<id>",
# "t6:<промпт>:<id>:correct_refusal", "t6:<промпт>:<id>:ungrounded",
# "t7:<сессия>:<шаг>:<вариант>".
# ---------------------------------------------------------------
JUDGMENTS = {
    # --- Задание 1: баллы LLM 0-2 (просмотрены полные ответы) ---
    # Сокращённый индекс, группа A (ответы есть в первых 5 фрагментах): всё верно
    "t1:A:first5:Почему, по мнению авторов, простое увеличение размера языковой модели не делает её лучше в следовании намерению пользователя?": 2,
    "t1:A:first5:Модель какого размера (InstructGPT) оказалась предпочтительнее выводов GPT-3 со 175 млрд параметров?": 2,
    "t1:A:first5:С помощью какого подхода авторы дообучали GPT-3 до модели InstructGPT?": 2,
    # Сокращённый индекс, группа B: два выдуманных ответа и один необоснованный отказ
    "t1:B:first5:Какие три вида промптов просили написать разметчиков для начального обучения InstructGPT?": 0,
    "t1:B:first5:Как часто асессоры предпочитали ответы InstructGPT 175B ответам моделей FLAN и T0?": 0,
    "t1:B:first5:Сколько подрядчиков наняли для разметки данных и через какие платформы?": 0,
    # Полный индекс: всё верно, кроме FLAN/T0 — названо только одно из двух чисел
    "t1:A:full:Почему, по мнению авторов, простое увеличение размера языковой модели не делает её лучше в следовании намерению пользователя?": 2,
    "t1:A:full:Модель какого размера (InstructGPT) оказалась предпочтительнее выводов GPT-3 со 175 млрд параметров?": 2,
    "t1:A:full:С помощью какого подхода авторы дообучали GPT-3 до модели InstructGPT?": 2,
    "t1:B:full:Какие три вида промптов просили написать разметчиков для начального обучения InstructGPT?": 2,
    "t1:B:full:Как часто асессоры предпочитали ответы InstructGPT 175B ответам моделей FLAN и T0?": 1,
    "t1:B:full:Сколько подрядчиков наняли для разметки данных и через какие платформы?": 2,

    # --- Задание 5: подтверждено ли утверждение текстом процитированного фрагмента ---
    # Вопрос 1: ссылки нет вовсе -> подтверждения нет
    "t5:1": False,
    # Вопрос 2: фрагмент 34 (стр. 7) содержит «hired a team of about 40 contractors on Upwork and through ScaleAI»
    "t5:2": True,
    # Вопрос 3: фрагмент 25 (стр. 6) — таблица 1 с «Generation 45.6%»
    "t5:3": True,
    # Вопрос 4: фрагмент 42 (стр. 9) содержит фразу про per-token KL penalty
    "t5:4": True,
    # Вопрос 5: фрагмент 59 (стр. 13) содержит «78 ±4%» и «79 ± 4%»
    "t5:5": True,

    # --- Задание 6: баллы 0-2 за вопросы с ответом ---
    "t6:base:1": 2, "t6:base:4": 2, "t6:base:5": 2, "t6:base:7": 2,
    "t6:base:9": 1,   # названо только 79 ± 4%, число для FLAN (78 ± 4%) потеряно
    "t6:strict:1": 2, "t6:strict:4": 2, "t6:strict:5": 2, "t6:strict:7": 2, "t6:strict:9": 2,
    # --- Задание 6: вопросы без ответа ---
    "t6:base:N1:ungrounded": True,   # выдумал «4.9 и 60 петафлопс/с-дней»
    "t6:base:N2:correct_refusal": True,
    "t6:base:N3:correct_refusal": True,
    "t6:base:N4:correct_refusal": True,
    "t6:base:N5:correct_refusal": True,
    "t6:strict:N1:correct_refusal": True,
    "t6:strict:N2:correct_refusal": True,
    "t6:strict:N3:correct_refusal": True,
    "t6:strict:N4:correct_refusal": True,
    "t6:strict:N5:correct_refusal": True,

    # --- Задание 7: баллы 0-2 за ответы в диалогах ---
    # Диалог 1
    "t7:dialog-1:1:Б": 2,   # верное определение InstructGPT
    "t7:dialog-1:2:А": 1,   # суть есть (промпты, RM, PPO), но неточности («RM для фильтрации»)
    "t7:dialog-1:2:Б": 1,   # поверхностно: нет структуры трёх этапов, лишнее про few-shot
    "t7:dialog-1:3:А": 2,   # размеры верны (ответ восстановлен из истории)
    "t7:dialog-1:3:Б": 2,
    # Диалог 2
    "t7:dialog-2:1:Б": 2,
    "t7:dialog-2:2:А": 1,   # данные названы, но путано (пропущены ранжирования разметчиков)
    "t7:dialog-2:2:Б": 1,   # перепутаны демонстрации (SFT) и ранжирования (RM)
    "t7:dialog-2:3:А": 2,
    "t7:dialog-2:3:Б": 2,
    # Диалог 3
    "t7:dialog-3:1:Б": 2,
    "t7:dialog-3:2:А": 2,
    "t7:dialog-3:2:Б": 2,
    "t7:dialog-3:3:А": 0,   # «Classification» в топ-3 — неверно (Open QA, Brainstorming)
    "t7:dialog-3:3:Б": 2,
}

In [6]:
# Базовый конвейер: строим полный индекс и задаём один контрольный вопрос
if not SKIP_EXPERIMENTS:
    index_full = get_index("full_1000_200", chunks)
    demo = rag("Какие три этапа включает метод обучения InstructGPT?", index_full)
else:
    print("Эксперименты пропущены (SKIP_EXPERIMENTS=1), читаем сохранённые результаты.")

Индекс 'full_1000_200': 246 фрагментов, построен за 73.3 с
Вопрос: Какие три этапа включает метод обучения InstructGPT?
Поисковый запрос: Какие три этапа включает метод обучения InstructGPT?
Найденные фрагменты:
  - Фрагмент 9, стр. 3: Figure 2: A diagram illustrating the three steps of our method: (1) supervised ﬁne-tuning (SFT), (2) reward model (RM) t...
  - Фрагмент 8, стр. 2: further in Section 5.2. We call the resulting models InstructGPT. We mainly evaluate our models by having our labelers r...
  - Фрагмент 2, стр. 1: submitted through the OpenAI API, we collect a dataset of labeler demonstrations of the desired model behavior, which we...
Ответ модели: Метод обучения InstructGPT включает три этапа: 
1. Сверхтонкая тюнинг (supervised fine-tuning),
2. Обучение вознаграждениям (reward model training),
3. Обучение через метод проксимальной стратегии оптимизации (reinforcement learning via proximal policy optimization, PPO).
---------------------------------------------------------

## Банк эталонных вопросов

Десять вопросов с эталонными ответами, цитатами и страницами (порядковые страницы PDF). Набор зафиксирован до экспериментов и дальше не меняется. Отдельно — пять тематически близких вопросов, ответа на которые в документе нет (проверено по полному тексту).

In [7]:
BANK = [
 {"id": 1, "q": "Какие три этапа включает метод обучения InstructGPT?",
  "answer": "SFT (supervised fine-tuning), обучение модели награды (RM) и обучение с подкреплением через PPO по этой модели награды.",
  "quote": "the three steps of our method: (1) supervised fine-tuning (SFT), (2) reward model (RM) training, and (3) reinforcement learning via proximal policy optimization (PPO)",
  "pages": [3]},
 {"id": 2, "q": "Модели каких размеров обучались в работе и на какой архитектуре?",
  "answer": "1.3B, 6B и 175B параметров; все модели используют архитектуру GPT-3.",
  "quote": "(1.3B, 6B, and 175B parameters), and all of our models use the GPT-3 architecture",
  "pages": [3]},
 {"id": 3, "q": "Сколько подрядчиков наняли для разметки данных и через какие платформы?",
  "answer": "Около 40 подрядчиков, через Upwork и ScaleAI.",
  "quote": "hired a team of about 40 contractors on Upwork and through ScaleAI",
  "pages": [18]},
 {"id": 4, "q": "Какая категория запросов занимает наибольшую долю в датасете API-промптов и сколько процентов она составляет?",
  "answer": "Generation (генерация) — 45.6%.",
  "quote": "Generation 45.6%",
  "pages": [6]},
 {"id": 5, "q": "Какие три вида промптов просили написать разметчиков для начального обучения InstructGPT?",
  "answer": "Plain (произвольная задача), Few-shot (инструкция с парами запрос/ответ) и User-based (по сценариям из заявок в листе ожидания API).",
  "quote": "we asked labelers to write three kinds of prompts",
  "pages": [7]},
 {"id": 6, "q": "Что показали результаты InstructGPT на бенчмарке TruthfulQA по сравнению с GPT-3?",
  "answer": "InstructGPT генерирует правдивые и информативные ответы примерно в два раза чаще, чем GPT-3.",
  "quote": "instructgpt generates truthful and informative answers about twice as often as gpt-3",
  "pages": [3]},
 {"id": 7, "q": "Сколько обучающих промптов вошло в SFT-датасет?",
  "answer": "Около 13 тысяч (12 725 в обучающей части).",
  "quote": "the sft dataset contains about 13k training prompts",
  "pages": [7]},
 {"id": 8, "q": "Что добавляют к цели оптимизации на этапе PPO, чтобы избежать переоптимизации модели награды?",
  "answer": "Потокеновый KL-штраф (KL penalty) относительно SFT-модели на каждом токене.",
  "quote": "we add a per-token kl penalty from the sft model at each token",
  "pages": [9]},
 {"id": 9, "q": "Как часто асессоры предпочитали ответы InstructGPT 175B ответам моделей FLAN и T0?",
  "answer": "FLAN — в 78 ± 4% случаев, T0 — в 79 ± 4% случаев.",
  "quote": "preferred over our flan model 78 ±4% of the time and over our t0 model 79 ± 4%",
  "pages": [13]},
 {"id": 10, "q": "Что авторы называют «alignment tax» и каким способом его удаётся снизить?",
  "answer": "Снижение качества на публичных NLP-датасетах при PPO-обучении; снижается смешиванием PPO-обновлений с обновлениями правдоподобия предобучающего распределения (PPO-ptx).",
  'quote': 'it suffers from an "alignment tax", as its performance on several public nlp datasets decreases',
  "pages": [14]},
]

NOANSWER = [
 {"id": "N1", "q": "Сколько GPU-часов потребовалось на полное обучение модели InstructGPT 175B?",
  "why": "Вычислительный бюджет обучения в статье не сообщается."},
 {"id": "N2", "q": "Какие результаты InstructGPT показал на математическом бенчмарке GSM8K?",
  "why": "Бенчмарк GSM8K в статье не используется."},
 {"id": "N3", "q": "Как InstructGPT сравнивается по качеству с моделью GPT-4?",
  "why": "GPT-4 в статье не упоминается (статья 2022 года)."},
 {"id": "N4", "q": "Кто был руководителем проекта InstructGPT в OpenAI?",
  "why": "Руководитель проекта в тексте не назван."},
 {"id": "N5", "q": "Какова была стоимость обучения InstructGPT в долларах?",
  "why": "Стоимость обучения в статье не приводится."},
]

for item in BANK:
    print(f"[{item['id']:>2}] стр. {str(item['pages']):>6} | {item['q']}")
print()
for item in NOANSWER:
    print(f"[{item['id']}] без ответа | {item['q']}")

[ 1] стр.    [3] | Какие три этапа включает метод обучения InstructGPT?
[ 2] стр.    [3] | Модели каких размеров обучались в работе и на какой архитектуре?
[ 3] стр.   [18] | Сколько подрядчиков наняли для разметки данных и через какие платформы?
[ 4] стр.    [6] | Какая категория запросов занимает наибольшую долю в датасете API-промптов и сколько процентов она составляет?
[ 5] стр.    [7] | Какие три вида промптов просили написать разметчиков для начального обучения InstructGPT?
[ 6] стр.    [3] | Что показали результаты InstructGPT на бенчмарке TruthfulQA по сравнению с GPT-3?
[ 7] стр.    [7] | Сколько обучающих промптов вошло в SFT-датасет?
[ 8] стр.    [9] | Что добавляют к цели оптимизации на этапе PPO, чтобы избежать переоптимизации модели награды?
[ 9] стр.   [13] | Как часто асессоры предпочитали ответы InstructGPT 175B ответам моделей FLAN и T0?
[10] стр.   [14] | Что авторы называют «alignment tax» и каким способом его удаётся снизить?

[N1] без ответа | Сколько GPU-часов по

## Задание 1. Полнота поискового индекса

Два индекса: только первые 5 фрагментов (стр. 1–2) и все фрагменты. Разбиение и модель эмбеддингов одинаковые, k=3. Три вопроса группы A — по началу документа, три вопроса группы B — по разделам вне первых пяти фрагментов. Ниже проверяется, что цитаты группы B действительно не входят в первые пять фрагментов.

In [24]:
T1_A = [
 {"q": "Почему, по мнению авторов, простое увеличение размера языковой модели не делает её лучше в следовании намерению пользователя?",
  "answer": "Большие модели могут генерировать неправдивые, токсичные или бесполезные ответы: размер сам по себе не учит следовать намерению пользователя (модели «не выровнены»).",
  "quote": "making language models bigger does not inherently make them better at following a user's intent",
  "pages": [1]},
 {"q": "Модель какого размера (InstructGPT) оказалась предпочтительнее выводов GPT-3 со 175 млрд параметров?",
  "answer": "Модель с 1.3 млрд параметров (1.3B) — несмотря на в 100 раз меньшее число параметров.",
  "quote": "outputs from the 1.3b parameter instructgpt model are preferred to outputs from the 175b gpt-3",
  "pages": [1]},
 {"q": "С помощью какого подхода авторы дообучали GPT-3 до модели InstructGPT?",
  "answer": "Сначала обучение с учителем (SFT) на демонстрациях разметчиков, затем обучение с подкреплением по человеческой обратной связи (RLHF) на ранжированиях выводов.",
  "quote": "reinforcement learning from human feedback",
  "pages": [1]},
]
T1_B = [BANK[4], BANK[8], BANK[2]]  # стр. 7, 13, 18

first5_text = normalize(" ".join(d.page_content for d in chunks[:5]))
print("Проверка покрытия первыми 5 фрагментами (стр. 1–2):")
for q in T1_A:
    print("  A, цитата входит в первые 5 фрагментов:", normalize(q["quote"]) in first5_text, "|", q["q"][:60])
for q in T1_B:
    print("  B, цитата входит в первые 5 фрагментов:", normalize(q["quote"]) in first5_text, "|", q["q"][:60])

Проверка покрытия первыми 5 фрагментами (стр. 1–2):
  A, цитата входит в первые 5 фрагментов: True | Почему, по мнению авторов, простое увеличение размера языков
  A, цитата входит в первые 5 фрагментов: True | Модель какого размера (InstructGPT) оказалась предпочтительн
  A, цитата входит в первые 5 фрагментов: True | С помощью какого подхода авторы дообучали GPT-3 до модели In
  B, цитата входит в первые 5 фрагментов: False | Какие три вида промптов просили написать разметчиков для нач
  B, цитата входит в первые 5 фрагментов: False | Как часто асессоры предпочитали ответы InstructGPT 175B отве
  B, цитата входит в первые 5 фрагментов: False | Сколько подрядчиков наняли для разметки данных и через какие


In [25]:
if not SKIP_EXPERIMENTS:
    idx_first5 = get_index("first5_1000_200", chunks[:5])
    idx_full = get_index("full_1000_200", chunks)

    t1_results = []
    for group, qs in (("A", T1_A), ("B", T1_B)):
        for q in qs:
            for iname, idx in (("first5", idx_first5), ("full", idx_full)):
                r = rag(q["q"], idx, verbose=True)
                r["group"] = group
                r["index"] = iname
                r["hit"] = quote_hit(r["retrieved"], q["quote"])
                r["page_hit"] = page_hit(r["retrieved"], q["pages"])
                t1_results.append(r)
    save_results("task1.json", t1_results)
else:
    t1_results = load_results("task1.json")
print("Всего прогонов:", len(t1_results))

Вопрос: Почему, по мнению авторов, простое увеличение размера языковой модели не делает её лучше в следовании намерению пользователя?
Поисковый запрос: Почему, по мнению авторов, простое увеличение размера языковой модели не делает её лучше в следовании намерению пользователя?
Найденные фрагменты:
  - Фрагмент 1, стр. 1: Training language models to follow instructions with human feedback Long Ouyang∗ Jeff Wu∗ Xu Jiang∗ Diogo Almeida∗ Carro...
  - Фрагмент 3, стр. 1: show that ﬁne-tuning with human feedback is a promising direction for aligning language models with human intent. 1 Intr...
  - Фрагмент 5, стр. 2: Brown et al., 2020; Fedus et al., 2021; Rae et al., 2021; Thoppilan et al., 2022). Thus, we say that the language modeli...
Ответ модели: По мнению авторов, простое увеличение размера языковой модели не делает её лучше в следовании намерению пользователя, потому что эти модели могут генерировать неуважительный или несвоевременный текст, лгать или не следовать инструкциям пользов

In [27]:
# Сводная таблица задания 1. Баллы LLM (0-2) выставлены вручную после
# просмотра ответов (results/judgments.json, ключи "t1:<группа>:<индекс>:<вопрос>").
J = JUDGMENTS or load_judgments()

def t1_stats(results):
    stats = {}
    for iname in ("first5", "full"):
        rows = [r for r in results if r["index"] == iname]
        hitA = sum(r["hit"] for r in rows if r["group"] == "A")
        hitB = sum(r["hit"] for r in rows if r["group"] == "B")
        n = len(rows)
        hit_rate = sum(r["hit"] for r in rows) / n * 100
        scores = [J.get(f"t1:{r['group']}:{iname}:{r['question']}", None) for r in rows]
        known = [s for s in scores if s is not None]
        avg = sum(known) / len(known) if known else float("nan")
        stats[iname] = (hitA, hitB, hit_rate, avg)
    return stats

stats = t1_stats(t1_results)
print(f"{'Индекс':<12} | {'Начало (из 3)':<13} | {'Далее (из 3)':<12} | {'Hit Rate@3':<10} | {'Средний балл':<12}")
print("-" * 75)
for iname, label in (("first5", "Первые 5"), ("full", "Полный")):
    hitA, hitB, hr, avg = stats[iname]
    print(f"{label:<12} | {hitA:<13} | {hitB:<12} | {hr:>6.0f}%   | {avg:>4.1f}")

print("\nРазбор ошибок сокращённого индекса:")
for r in t1_results:
    if r["index"] == "first5" and not r["hit"]:
        print(f"\n* {r['question']}")
        print("  Найдено:", [(d['chunk_id'], d['page_pdf']) for d in r['retrieved']])
        print("  Ответ LLM:", r["answer"][:300])

Индекс       | Начало (из 3) | Далее (из 3) | Hit Rate@3 | Средний балл
---------------------------------------------------------------------------
Первые 5     | 3             | 0            |     50%   |  1.0
Полный       | 3             | 3            |    100%   |  1.8

Разбор ошибок сокращённого индекса:

* Какие три вида промптов просили написать разметчиков для начального обучения InstructGPT?
  Найдено: [(2, 1), (4, 2), (1, 1)]
  Ответ LLM: Разметчикам просили написать три вида промптов для начального обучения InstructGPT: стандартные промпты, промпты, поданные через API OpenAI, и промпты, созданные автоматически.

* Как часто асессоры предпочитали ответы InstructGPT 175B ответам моделей FLAN и T0?
  Найдено: [(4, 2), (2, 1), (1, 1)]
  Ответ LLM: Ассессоры предпочитали ответы InstructGPT 175B ответам моделей FLAN и T0 в 60% случаях.

* Сколько подрядчиков наняли для разметки данных и через какие платформы?
  Найдено: [(1, 1), (2, 1), (4, 2)]
  Ответ LLM: В статье не указано ско

## Задание 2. Размер фрагментов

`chunk_size` = 500, 1000 и 2000 при `chunk_overlap=100`, k=3, полный индекс. Для каждого размера фрагменты и индекс создаются заново, вопросы — общий банк из десяти. Успех: эталонная цитата целиком содержится хотя бы в одном фрагменте из top-3.

In [11]:
if not SKIP_EXPERIMENTS:
    t2_results = {}
    for cs in (500, 1000, 2000):
        ch = make_chunks(cs, 100)
        idx = get_index(f"size{cs}_ov100", ch)
        rows = []
        for q in BANK:
            retrieved = retrieve(idx, q["q"])
            ret_meta = [{"chunk_id": d.metadata["chunk_id"],
                         "page_pdf": d.metadata["page_pdf"],
                         "text": d.page_content} for d in retrieved]
            rows.append({"id": q["id"], "q": q["q"],
                         "hit": quote_hit(ret_meta, q["quote"]),
                         "page_hit": page_hit(ret_meta, q["pages"]),
                         "retrieved": [(d["chunk_id"], d["page_pdf"]) for d in ret_meta]})
        t2_results[str(cs)] = {"n_chunks": len(ch), "rows": rows}
    save_results("task2.json", t2_results)
else:
    t2_results = load_results("task2.json")

print(f"{'Размер':<8} | {'Фрагментов':<10} | {'Успешных из 10':<14} | {'Hit Rate@3':<10}")
print("-" * 55)
for cs in ("500", "1000", "2000"):
    cfg = t2_results[cs]
    hits = sum(r["hit"] for r in cfg["rows"])
    print(f"{cs:<8} | {cfg['n_chunks']:<10} | {hits:<14} | {hits/10*100:>5.0f}%")

print("\nДетализация по вопросам (hit по цитате):")
print(f"{'id':<4} | {'500':<5} | {'1000':<5} | {'2000':<5}")
for q in BANK:
    row = [t2_results[cs]["rows"][q["id"]-1]["hit"] for cs in ("500", "1000", "2000")]
    print(f"{q['id']:<4} | {str(row[0]):<5} | {str(row[1]):<5} | {str(row[2]):<5}")

Индекс 'size500_ov100': 476 фрагментов, построен за 10.5 с
Индекс 'size1000_ov100': 228 фрагментов, построен за 7.2 с
Индекс 'size2000_ov100': 128 фрагментов, построен за 4.8 с
Размер   | Фрагментов | Успешных из 10 | Hit Rate@3
-------------------------------------------------------
500      | 476        | 7              |    70%
1000     | 228        | 7              |    70%
2000     | 128        | 7              |    70%

Детализация по вопросам (hit по цитате):
id   | 500   | 1000  | 2000 
1    | False | True  | True 
2    | False | False | False
3    | True  | False | False
4    | True  | True  | True 
5    | True  | True  | True 
6    | True  | True  | False
7    | True  | True  | True 
8    | True  | True  | True 
9    | True  | True  | True 
10   | False | False | True 


## Задание 3. Перекрытие фрагментов

`chunk_size=1000`, перекрытие 0, 100 и 200, полный индекс, k=3.

Нюанс: рекурсивный сплиттер режет по абзацам и предложениям, поэтому ни одна эталонная цитата при overlap=0 не оказалась разрезана. Берём пять вопросов, чьи цитаты ближе всего к границе фрагмента, — именно им перекрытие должно помогать.

«Почти одинаковыми» считаем два результата top-3 с долей общих слов (Jaccard) ≥ 0.5.

In [12]:
ch_ov0 = make_chunks(1000, 0)

def quote_boundary_distance(quote, docs):
    """Расстояние (в символах) от цитаты до ближайшего края фрагмента,
    в который она попала целиком; None — если цитата разрезана границей."""
    nq = normalize(quote)
    for d in docs:
        t = normalize(d.page_content)
        i = t.find(nq)
        if i >= 0:
            return min(i, len(t) - (i + len(nq)))
    return None

distances = [(quote_boundary_distance(q["quote"], ch_ov0), q) for q in BANK]
split_quotes = [q for d, q in distances if d is None]
near = sorted((d, q) for d, q in distances if d is not None)
T3_QS = [q for _, q in near[:5]]

print("Разрезанных границей цитат при overlap=0:", len(split_quotes))
print("Пять вопросов, чьи цитаты ближе всего к границам фрагментов:")
for d, q in near[:5]:
    print(f"  [{q['id']:>2}] {d:>4} символов до края | {q['q'][:65]}")

def jaccard(a, b):
    A, B = set(normalize(a).split()), set(normalize(b).split())
    return len(A & B) / max(1, len(A | B))

def has_near_duplicates(retrieved, threshold=0.5):
    for d1, d2 in combinations(retrieved, 2):
        if jaccard(d1["text"], d2["text"]) >= threshold:
            return True
    return False

Разрезанных границей цитат при overlap=0: 0
Пять вопросов, чьи цитаты ближе всего к границам фрагментов:
  [ 1]   33 символов до края | Какие три этапа включает метод обучения InstructGPT?
  [ 9]   84 символов до края | Как часто асессоры предпочитали ответы InstructGPT 175B ответам м
  [ 4]   87 символов до края | Какая категория запросов занимает наибольшую долю в датасете API-
  [ 7]  155 символов до края | Сколько обучающих промптов вошло в SFT-датасет?
  [10]  159 символов до края | Что авторы называют «alignment tax» и каким способом его удаётся 


In [13]:
if not SKIP_EXPERIMENTS:
    t3_results = {}
    for ov in (0, 100, 200):
        ch = make_chunks(1000, ov)
        idx = get_index(f"size1000_ov{ov}", ch)
        rows = []
        for q in T3_QS:
            retrieved = retrieve(idx, q["q"])
            ret_meta = [{"chunk_id": d.metadata["chunk_id"],
                         "page_pdf": d.metadata["page_pdf"],
                         "text": d.page_content} for d in retrieved]
            rows.append({"id": q["id"], "q": q["q"],
                         "hit": quote_hit(ret_meta, q["quote"]),
                         "dup": has_near_duplicates(ret_meta),
                         "retrieved": [(d["chunk_id"], d["page_pdf"]) for d in ret_meta]})
        t3_results[str(ov)] = {"n_chunks": len(ch), "rows": rows}
    save_results("task3.json", t3_results)
else:
    t3_results = load_results("task3.json")

print(f"{'Перекрытие':<10} | {'Фрагментов':<10} | {'Успешных из 5':<13} | {'С повторами в top-3':<19}")
print("-" * 60)
for ov in ("0", "100", "200"):
    cfg = t3_results[ov]
    hits = sum(r["hit"] for r in cfg["rows"])
    dups = sum(r["dup"] for r in cfg["rows"])
    print(f"{ov:<10} | {cfg['n_chunks']:<10} | {hits:<13} | {dups:<19}")

print("\nДетализация:")
for r0, r100, r200 in zip(t3_results["0"]["rows"], t3_results["100"]["rows"], t3_results["200"]["rows"]):
    print(f"[{r0['id']}] hit: 0->{r0['hit']}, 100->{r100['hit']}, 200->{r200['hit']} | "
          f"dup: 0->{r0['dup']}, 100->{r100['dup']}, 200->{r200['dup']} | {r0['q'][:55]}")

Индекс 'size1000_ov0': 217 фрагментов, построен за 5.2 с
Индекс 'size1000_ov200': 246 фрагментов, построен за 6.0 с
Перекрытие | Фрагментов | Успешных из 5 | С повторами в top-3
------------------------------------------------------------
0          | 217        | 4             | 0                  
100        | 228        | 4             | 0                  
200        | 246        | 5             | 0                  

Детализация:
[1] hit: 0->True, 100->True, 200->True | dup: 0->False, 100->False, 200->False | Какие три этапа включает метод обучения InstructGPT?
[9] hit: 0->True, 100->True, 200->True | dup: 0->False, 100->False, 200->False | Как часто асессоры предпочитали ответы InstructGPT 175B
[4] hit: 0->True, 100->True, 200->True | dup: 0->False, 100->False, 200->False | Какая категория запросов занимает наибольшую долю в дат
[7] hit: 0->True, 100->True, 200->True | dup: 0->False, 100->False, 200->False | Сколько обучающих промптов вошло в SFT-датасет?
[10] hit: 0->False, 100-

## Задание 4. Векторный поиск и мешок слов

По тем же фрагментам: бинарный мешок слов + косинусное сходство против эмбеддингов. Словарь строится только по документу; если запрос даёт нулевой вектор — фиксируем отсутствие лексических совпадений, а не считаем выдачу поиском.

Пять вопросов с терминами документа (англ.) и пять переформулировок с тем же смыслом (рус.). k=3.

In [14]:
T4_PAIRS = [
 {"exact": "What are the three steps of the InstructGPT training method?",
  "para": "Опиши этапы обучения модели InstructGPT.",
  "ref": BANK[0]},
 {"exact": "How many contractors were hired to label the data?",
  "para": "Сколько человек размечали данные для этого исследования?",
  "ref": BANK[2]},
 {"exact": "What KL penalty is added during PPO training?",
  "para": "Как авторы не дали обучаемой политике слишком далеко уйти от исходной SFT-модели?",
  "ref": BANK[7]},
 {"exact": "Which use-case category has the largest share in the API prompt dataset?",
  "para": "Какие запросы пользователи отправляют в API чаще всего?",
  "ref": BANK[3]},
 {"exact": "How often does InstructGPT generate truthful and informative answers on TruthfulQA compared to GPT-3?",
  "para": "Стала ли выровненная модель честнее базовой?",
  "ref": BANK[5]},
]

def tokenize(s):
    return re.findall(r"[a-z0-9а-яё]+", normalize(s))

# Словарь — только по фрагментам документа
docs_tokens = [tokenize(d.page_content) for d in chunks]
vocab = sorted(set(w for t in docs_tokens for w in t))
vidx = {w: i for i, w in enumerate(vocab)}
print("Размер словаря:", len(vocab))

def bow_vec(tokens):
    v = np.zeros(len(vocab), dtype=np.float64)
    for w in tokens:
        if w in vidx:
            v[vidx[w]] = 1.0          # бинарное представление: 1, если слово встречается
    return v

BOW_D = np.vstack([bow_vec(t) for t in docs_tokens])
BOW_NORM = np.linalg.norm(BOW_D, axis=1)

def bow_search(query, k=K):
    """Бинарный мешок слов + косинусное сходство. Нулевой вектор -> нет совпадений."""
    qv = bow_vec(tokenize(query))
    if qv.sum() == 0:
        return [], True
    sims = (BOW_D @ qv) / (BOW_NORM * np.linalg.norm(qv) + 1e-12)
    top = np.argsort(-sims)[:k]
    return ([{"chunk_id": chunks[i].metadata["chunk_id"],
              "page_pdf": chunks[i].metadata["page_pdf"],
              "text": chunks[i].page_content,
              "score": float(sims[i])} for i in top], False)

Размер словаря: 4714


In [15]:
if not SKIP_EXPERIMENTS:
    idx_full = get_index("full_1000_200", chunks)
    t4_results = []
    for pair in T4_PAIRS:
        for kind, qtext in (("exact", pair["exact"]), ("para", pair["para"])):
            # бинарный мешок слов
            bow_ret, zero_vec = bow_search(qtext)
            # плотные эмбеддинги (те же фрагменты)
            dense_ret_docs = retrieve(idx_full, qtext)
            dense_ret = [{"chunk_id": d.metadata["chunk_id"],
                          "page_pdf": d.metadata["page_pdf"],
                          "text": d.page_content} for d in dense_ret_docs]
            t4_results.append({
                "kind": kind, "q": qtext, "ref_id": pair["ref"]["id"],
                "bow_zero_vector": zero_vec,
                "bow_hit": (not zero_vec) and quote_hit(bow_ret, pair["ref"]["quote"]),
                "bow_pages": [d["page_pdf"] for d in bow_ret],
                "dense_hit": quote_hit(dense_ret, pair["ref"]["quote"]),
                "dense_pages": [d["page_pdf"] for d in dense_ret],
            })
    save_results("task4.json", t4_results)
else:
    t4_results = load_results("task4.json")

def t4_rate(rows):
    n = len(rows)
    bow = sum(r["bow_hit"] for r in rows)
    dense = sum(r["dense_hit"] for r in rows)
    return bow, dense, n

ex = [r for r in t4_results if r["kind"] == "exact"]
pa = [r for r in t4_results if r["kind"] == "para"]
b_e, d_e, n_e = t4_rate(ex)
b_p, d_p, n_p = t4_rate(pa)
print(f"{'Метод':<22} | {'Термины (из 5)':<14} | {'Переформулировки (из 5)':<23} | Hit Rate@3")
print("-" * 85)
print(f"{'Бинарный мешок слов':<22} | {b_e:<14} | {b_p:<23} | {b_e/n_e*100:.0f}% / {b_p/n_p*100:.0f}%")
print(f"{'Плотные эмбеддинги':<22} | {d_e:<14} | {d_p:<23} | {d_e/n_e*100:.0f}% / {d_p/n_p*100:.0f}%")

print("\nДетализация (стр. — страницы top-3):")
for r in t4_results:
    zero = ", НУЛЕВОЙ ВЕКТОР" if r["bow_zero_vector"] else ""
    print(f"[{r['kind']:<5}] {r['q'][:60]}")
    print(f"        BoW: hit={r['bow_hit']} стр.={r['bow_pages']}{zero} | dense: hit={r['dense_hit']} стр.={r['dense_pages']}")

Метод                  | Термины (из 5) | Переформулировки (из 5) | Hit Rate@3
-------------------------------------------------------------------------------------
Бинарный мешок слов    | 3              | 0                       | 60% / 0%
Плотные эмбеддинги     | 4              | 3                       | 80% / 60%

Детализация (стр. — страницы top-3):
[exact] What are the three steps of the InstructGPT training method?
        BoW: hit=True стр.=[3, 2, 26] | dense: hit=True стр.=[3, 2, 1]
[para ] Опиши этапы обучения модели InstructGPT.
        BoW: hit=False стр.=[62, 63, 15] | dense: hit=True стр.=[3, 2, 1]
[exact] How many contractors were hired to label the data?
        BoW: hit=False стр.=[36, 18, 33] | dense: hit=True стр.=[19, 7, 38]
[para ] Сколько человек размечали данные для этого исследования?
        BoW: hit=False стр.=[], НУЛЕВОЙ ВЕКТОР | dense: hit=True стр.=[38, 7, 33]
[exact] What KL penalty is added during PPO training?
        BoW: hit=True стр.=[9, 9, 57] | den

## Задание 5. Проверяемые ссылки на источники

В метаданных фрагмента: имя файла, порядковая страница PDF, уникальный id. Перед каждым фрагментом в контексте идёт заголовок `[Фрагмент N, файл, стр. M]`, промпт требует ссылок вида `[Фрагмент N, стр. M]` и запрещает выдумывать источники.

Проверка каждой ссылки: id среди переданных фрагментов; страница совпадает с метаданными; утверждение подтверждается текстом (вручную).

In [28]:
T5_QS = [BANK[1], BANK[2], BANK[3], BANK[7], BANK[8]]  # id 2,3,4,8,9

CITE_RE = re.compile(r"\[Фрагмент\s+(\d+)\s*,\s*стр\.\s*(\d+)\s*\]")

if not SKIP_EXPERIMENTS:
    idx_full = get_index("full_1000_200", chunks)
    t5_results = []
    for q in T5_QS:
        r = rag(q["q"], idx_full, prompt_template=CITATION_PROMPT, verbose=True)
        passed_ids = {d["chunk_id"]: d["page_pdf"] for d in r["retrieved"]}
        cites = [{"chunk_id": int(cid), "page": int(pg)} for cid, pg in CITE_RE.findall(r["answer"])]
        checks = []
        for c in cites:
            exists = c["chunk_id"] in passed_ids
            page_ok = exists and passed_ids[c["chunk_id"]] == c["page"]
            checks.append({**c, "exists": exists, "page_ok": page_ok})
        r["citations"] = cites
        r["checks"] = checks
        r["ref_id"] = q["id"]
        t5_results.append(r)
    save_results("task5.json", t5_results)
else:
    t5_results = load_results("task5.json")

J = JUDGMENTS or load_judgments()
print(f"{'Вопрос':<8} | {'Ссылки':<22} | {'ID существует':<13} | {'Страница верна':<14} | {'Утверждение подтверждено':<24}")
print("-" * 95)
for i, r in enumerate(t5_results, 1):
    ids = [(c["chunk_id"], c["page"]) for c in r["citations"]]
    ok_id = all(c["exists"] for c in r["checks"]) if r["checks"] else False
    ok_pg = all(c["page_ok"] for c in r["checks"]) if r["checks"] else False
    confirmed = J.get(f"t5:{i}", None)
    print(f"{i:<8} | {str(ids):<22} | {str(ok_id):<13} | {str(ok_pg):<14} | {str(confirmed):<24}")

print("\nОтветы модели:")
for i, r in enumerate(t5_results, 1):
    print(f"\n--- Вопрос {i}: {r['question']}")
    print(r["answer"])

Вопрос: Модели каких размеров обучались в работе и на какой архитектуре?
Поисковый запрос: Модели каких размеров обучались в работе и на какой архитектуре?
Найденные фрагменты:
  - Фрагмент 171, стр. 40: down to 10% of the original learning rate, with no learning rate warmup. For our 1.3B and 6B models, we use an LR of 9.6...
  - Фрагмент 174, стр. 41: Winogrande). This was mostly for historical reasons; we ﬁnd similar results when initializing the RM from the GPT-3 or S...
  - Фрагмент 170, стр. 40: All model architectures use the GPT-3 architecture (Brown et al., 2020). For the reward models and value functions, the ...
Ответ модели: Модели размером 1.3B, 6B и 175B обучались на архитектуре GPT-3 [Фрагмент 170, стр. 40], [Фрагмент 171, стр. 40], [Фрагмент 174, стр. 41].
--------------------------------------------------------------------------------
Вопрос: Сколько подрядчиков наняли для разметки данных и через какие платформы?
Поисковый запрос: Сколько подрядчиков наняли для разметки

## Задание 6. Отсутствие ответа в документе

Пять вопросов с ответом и пять тематически близких без ответа (отсутствие ответа проверено по полному тексту). Исходный промпт против строгого (отвечать только по контексту, при недостатке сведений — явный отказ). Модель, параметры и поиск одинаковые: оба промпта получают одни и те же заранее извлечённые фрагменты.

In [29]:
T6_ANS = [BANK[0], BANK[3], BANK[4], BANK[6], BANK[8]]   # id 1,4,5,7,9
T6_NO = NOANSWER

REFUSAL_MARKERS = ["нет информации", "не содержит", "недостаточно сведен",
                   "нет сведений", "не упомина", "не указан", "не привод",
                   "no information", "not mentioned", "does not contain",
                   "не сообщается", "не описан", "отсутствует"]

def is_refusal(text):
    t = normalize(text)
    return any(m in t for m in REFUSAL_MARKERS)

if not SKIP_EXPERIMENTS:
    idx_full = get_index("full_1000_200", chunks)
    t6_results = []
    for kind, qs in (("answer", T6_ANS), ("noanswer", T6_NO)):
        for q in qs:
            # один и тот же поиск для обоих промптов
            retrieved = retrieve(idx_full, q["q"])
            ret_meta = [{"chunk_id": d.metadata["chunk_id"],
                         "page_pdf": d.metadata["page_pdf"],
                         "text": d.page_content} for d in retrieved]
            context = format_context(retrieved)
            ans_base = ask_llm(BASE_PROMPT.format(context=context, question=q["q"]))
            ans_strict = ask_llm(STRICT_PROMPT.format(context=context, question=q["q"]))
            t6_results.append({
                "kind": kind, "id": q["id"], "q": q["q"],
                "retrieved": [(d["chunk_id"], d["page_pdf"]) for d in ret_meta],
                "hit": quote_hit(ret_meta, q["quote"]) if kind == "answer" else None,
                "ans_base": ans_base, "ans_strict": ans_strict,
                "refusal_base": is_refusal(ans_base),
                "refusal_strict": is_refusal(ans_strict),
            })
            print(f"[{kind}] {q['q'][:60]}")
            print("  исходный промпт :", ans_base[:200].replace("\n", " "))
            print("  строгий промпт  :", ans_strict[:200].replace("\n", " "))
            print("-" * 80)
    save_results("task6.json", t6_results)
else:
    t6_results = load_results("task6.json")
print("Прогонов:", len(t6_results))

[answer] Какие три этапа включает метод обучения InstructGPT?
  исходный промпт : Метод обучения InstructGPT включает три этапа:  1. Сверхтонкая тюнинг (supervised fine-tuning), 2. Обучение вознаграждениям (reward model training), 3. Обучение через оптимизацию стратегий (reinforcem
  строгий промпт  : Метод обучения InstructGPT включает три этапа:  1. Суффиксированная файн-тюнинг (SFT), 2. Обучение вознагражденияного моделей (RM), 3. Обучение через релаксационное управление с использованием метода 
--------------------------------------------------------------------------------
[answer] Какая категория запросов занимает наибольшую долю в датасете
  исходный промпт : Категория запросов "Generation" занимает наибольшую долю в датасете API-промптов и составляет 45,6% процентов.
  строгий промпт  : Category "Generation" занимает наибольшую долю в датасете API-промптов и составляет 45.6% данных.
--------------------------------------------------------------------------------
[answer] Какие 

In [30]:
# Баллы за ответы на вопросы с ответом (0-2) и вердикты по отказам
# заполнены вручную после просмотра results/task6.json (results/judgments.json).
J = JUDGMENTS or load_judgments()

for prompt_kind, label in (("base", "Исходный"), ("strict", "Строгий")):
    rows_a = [r for r in t6_results if r["kind"] == "answer"]
    scores = [J.get(f"t6:{prompt_kind}:{r['id']}", None) for r in rows_a]
    known = [s for s in scores if s is not None]
    avg = sum(known) / len(known) if known else float("nan")
    rows_n = [r for r in t6_results if r["kind"] == "noanswer"]
    refusals = sum(r[f"refusal_{prompt_kind}"] for r in rows_n)
    correct_refusals = sum(J.get(f"t6:{prompt_kind}:{r['id']}:correct_refusal", False) for r in rows_n)
    ungrounded = sum(J.get(f"t6:{prompt_kind}:{r['id']}:ungrounded", False) for r in rows_n)
    print(f"{label:<10} | средний балл (5 вопросов с ответом): {avg:.1f} | "
          f"корректных отказов из 5: {refusals} (авто) / {correct_refusals} (ручн.) | "
          f"неподтверждённых ответов из 5: {ungrounded}")

Исходный   | средний балл (5 вопросов с ответом): 1.8 | корректных отказов из 5: 3 (авто) / 4 (ручн.) | неподтверждённых ответов из 5: 1
Строгий    | средний балл (5 вопросов с ответом): 2.0 | корректных отказов из 5: 5 (авто) / 5 (ручн.) | неподтверждённых ответов из 5: 0


## Задание 7. Поддержка уточняющих вопросов

История хранится по session_id; в модель уходят последние 3 завершённых обмена и текущий вопрос; историю можно очистить.

- Вариант А: поиск по сырой реплике, история только в промпте генерации.
- Вариант Б: LLM сначала переформулирует уточнение в самостоятельный запрос, затем поиск и генерация.

Три диалога, в каждом первый вопрос и два уточнения. Для уточнений выводятся исходная реплика, фактический запрос для поиска, найденные фрагменты и ответ.

In [31]:
REWRITE_PROMPT = (
    "Ниже дана история диалога и уточняющий вопрос пользователя.\n\n"
    "История диалога:\n{history}\n\n"
    "Уточняющий вопрос: {question}\n\n"
    "Переформулируй уточняющий вопрос в самостоятельный поисковый запрос, "
    "раскрыв все ссылки на предыдущие реплики (например, «его», «это», «они»). "
    "Не отвечай на вопрос и не добавляй сведений, которых нет в истории. "
    "Выведи только самостоятельный запрос, без пояснений.\n\n"
    "Самостоятельный запрос:"
)

ANSWER_WITH_HISTORY_PROMPT = (
    "Ты — ассистент, который отвечает на вопросы по научной статье. "
    "Учитывай историю диалога для понимания вопроса, но факты бери только "
    "из фрагментов статьи.\n\n"
    "История диалога:\n{history}\n\n"
    "Фрагменты статьи:\n{context}\n\n"
    "Вопрос: {question}\n\n"
    "Дай краткий точный ответ на русском языке, опираясь на фрагменты.\n"
    "Ответ:"
)

# Хранилище историй: session_id -> список обменов (вопрос, ответ)
SESSIONS = {}

def get_history(session_id, n=3):
    """Последние n завершённых обменов данной сессии."""
    return SESSIONS.get(session_id, [])[-n:]

def history_text(session_id):
    h = get_history(session_id)
    if not h:
        return "(история пуста)"
    return "\n".join(f"Вопрос: {q}\nОтвет: {a}" for q, a in h)

def push_history(session_id, question, answer):
    SESSIONS.setdefault(session_id, []).append((question, answer))

def clear_history(session_id):
    SESSIONS.pop(session_id, None)

def rewrite_query(session_id, question):
    """Вариант Б: уточнение -> самостоятельный поисковый запрос."""
    if not get_history(session_id):
        return question
    out = ask_llm(REWRITE_PROMPT.format(
        history=history_text(session_id), question=question))
    # убираем возможные кавычки/префиксы
    out = out.strip().strip('"«»').split("\n")[0].strip()
    for prefix in ("Самостоятельный запрос:", "Запрос:", "Переформулированный вопрос:"):
        if out.startswith(prefix):
            out = out[len(prefix):].strip()
    return out or question

DIALOGS = [
 {"session": "dialog-1",
  "steps": [
    {"q": "Что такое InstructGPT?", "ref_quote": "we call the resulting models instructgpt", "pages": [1]},
    {"q": "Как его обучали?", "ref_quote": "the three steps of our method: (1) supervised fine-tuning (sft), (2) reward model (rm) training, and (3) reinforcement learning via proximal policy optimization (ppo)", "pages": [3]},
    {"q": "Модели каких размеров при этом использовали?", "ref_quote": "(1.3b, 6b, and 175b parameters), and all of our models use the gpt-3 architecture", "pages": [3]},
  ]},
 {"session": "dialog-2",
  "steps": [
    {"q": "Что такое модель награды (reward model) в этой статье?", "ref_quote": "reward model (rm) training", "pages": [3]},
    {"q": "На каких данных её обучали?", "ref_quote": "our rm dataset, with labeler rankings of model outputs used to train our rms", "pages": [7]},
    {"q": "А как потом её использовали при дообучении?", "ref_quote": "reinforcement learning via proximal policy optimization (ppo) on this reward model", "pages": [3]},
  ]},
 {"session": "dialog-3",
  "steps": [
    {"q": "Какая категория запросов самая частая в датасете API-промптов?", "ref_quote": "generation 45.6%", "pages": [6]},
    {"q": "А какая у неё доля в процентах?", "ref_quote": "generation 45.6%", "pages": [6]},
    {"q": "Какие категории ещё входят в топ-3?", "ref_quote": "open qa 12.4%", "pages": [6]},
  ]},
]

In [32]:
if not SKIP_EXPERIMENTS:
    idx_full = get_index("full_1000_200", chunks)
    SESSIONS.clear()
    t7_results = []
    for dialog in DIALOGS:
        sid = dialog["session"]
        for step_no, step in enumerate(dialog["steps"], 1):
            is_followup = step_no > 1
            # --- вариант Б: самостоятельный запрос по истории
            query_b = rewrite_query(sid, step["q"]) if is_followup else step["q"]
            ret_b = retrieve(idx_full, query_b)
            ctx_b = format_context(ret_b)
            ans_b = ask_llm(ANSWER_WITH_HISTORY_PROMPT.format(
                history=history_text(sid), context=ctx_b, question=step["q"]))
            rec = {"session": sid, "step": step_no, "question": step["q"],
                   "query_b": query_b,
                   "ret_b": [{"chunk_id": d.metadata["chunk_id"],
                              "page_pdf": d.metadata["page_pdf"],
                              "text": d.page_content} for d in ret_b],
                   "hit_b": quote_hit([{"text": d.page_content} for d in ret_b], step["ref_quote"]),
                   "ans_b": ans_b}
            # --- вариант А (только для уточнений): поиск по сырому вопросу
            if is_followup:
                ret_a = retrieve(idx_full, step["q"])
                ctx_a = format_context(ret_a)
                ans_a = ask_llm(ANSWER_WITH_HISTORY_PROMPT.format(
                    history=history_text(sid), context=ctx_a, question=step["q"]))
                rec.update({
                    "query_a": step["q"],
                    "ret_a": [{"chunk_id": d.metadata["chunk_id"],
                               "page_pdf": d.metadata["page_pdf"],
                               "text": d.page_content} for d in ret_a],
                    "hit_a": quote_hit([{"text": d.page_content} for d in ret_a], step["ref_quote"]),
                    "ans_a": ans_a})
            push_history(sid, step["q"], ans_b)
            t7_results.append(rec)
            print(f"[{sid} шаг {step_no}] {step['q']}")
            if is_followup:
                print(f"  А: запрос «{step['q']}» -> hit={rec['hit_a']}")
                print(f"     Ответ А: {rec['ans_a']}")
                print(f"  Б: запрос «{query_b}» -> hit={rec['hit_b']}")
            print(f"  Ответ Б: {ans_b}")
            print("-" * 80)
    save_results("task7.json", t7_results)
else:
    t7_results = load_results("task7.json")
print("Реплик обработано:", len(t7_results))

[dialog-1 шаг 1] Что такое InstructGPT?
  Ответ Б: InstructGPT - это модель, которая показала способность лучше следовать инструкциям по сравнению с GPT-3, несмотря на то, что InstructGPT имеет меньше параметров. InstructGPT также демонстриет улучшенную точность и уменьшенное количество ошибок, включая создание ложной информации и неверное понимание инструкций.
--------------------------------------------------------------------------------
[dialog-1 шаг 2] Как его обучали?
  А: запрос «Как его обучали?» -> hit=False
     Ответ А: Обучение InstructGPT проводилось с использованием предварительно написанных инструкций, которые создавали лейблеры. Лейблеры писали три типа инструкций: простые, с несколькими примерами и пользовательские, соответствующие заявкам из списка ожидаемых пользователей OpenAI API. Эти инструкции использовались для обучения базовых моделей на основе подкрепления, а затем для обучения InstructGPT с использованием RM (reinforcement model) на основе сравнений, предоста

In [21]:
# Таблица по уточнениям. Баллы ответов (0-2) — ручная оценка (judgments.json).
J = JUDGMENTS or load_judgments()
print(f"{'Диалог/шаг':<10} | {'Тип':<2} | {'Запрос для поиска':<52} | {'Фрагмент найден':<15} | {'Балл':<4}")
print("-" * 100)
for r in t7_results:
    if r["step"] == 1:
        key = f"t7:{r['session']}:{r['step']}:B"
        print(f"{r['session']}/{r['step']:<4} | {'Б':<2} | {r['query_b'][:52]:<52} | {str(r['hit_b']):<15} | {J.get(key, '-')}")
    else:
        for variant, qkey, hkey in (("А", "query_a", "hit_a"), ("Б", "query_b", "hit_b")):
            key = f"t7:{r['session']}:{r['step']}:{variant}"
            print(f"{r['session']}/{r['step']:<4} | {variant:<2} | {r[qkey][:52]:<52} | {str(r[hkey]):<15} | {J.get(key, '-')}")

imp = sum(1 for r in t7_results if r["step"] > 1 and r["hit_b"] and not r["hit_a"])
same = sum(1 for r in t7_results if r["step"] > 1 and r["hit_b"] == r["hit_a"])
print(f"\nУточнений, где вариант Б нашёл нужный фрагмент, а А — нет: {imp}; совпадений: {same}")

Диалог/шаг | Тип | Запрос для поиска                                    | Фрагмент найден | Балл
----------------------------------------------------------------------------------------------------
dialog-1/1    | Б  | Что такое InstructGPT?                               | False           | 2
dialog-1/2    | А  | Как его обучали?                                     | False           | 1
dialog-1/2    | Б  | Как обучали InstructGPT по сравнению с GPT-3?        | False           | 1
dialog-1/3    | А  | Модели каких размеров при этом использовали?         | False           | 2
dialog-1/3    | Б  | Какие размеры моделей были использованы при обучении | True            | 2
dialog-2/1    | Б  | Что такое модель награды (reward model) в этой стать | True            | 2
dialog-2/2    | А  | На каких данных её обучали?                          | False           | 1
dialog-2/2    | Б  | На каких данных была обучена модель награды (reward  | False           | 1
dialog-2/3    | А  | А как потом е

In [33]:
# Проверка семантики сессий
if not SKIP_EXPERIMENTS:
    print("История dialog-1 (последние 3 обмена):")
    for q, a in get_history("dialog-1"):
        print("  Q:", q[:60], "| A:", a[:60])
    print("\nНовая сессия 'dialog-new':", get_history("dialog-new"), "(независимый разговор)")
    clear_history("dialog-1")
    print("После очистки dialog-1:", get_history("dialog-1"), "(истории нет)")
    print("rewrite без истории возвращает вопрос как есть:",
          rewrite_query("dialog-1", "Как его обучали?"))
else:
    print("См. результаты выше (эксперименты пропущены).")

История dialog-1 (последние 3 обмена):
  Q: Что такое InstructGPT? | A: InstructGPT - это модель, которая показала способность лучше
  Q: Как его обучали? | A: Обучение InstructGPT проводилось путем его файн-тюнинга на д
  Q: Модели каких размеров при этом использовали? | A: В этом исследовании использовались модели с размерами параме

Новая сессия 'dialog-new': [] (независимый разговор)
После очистки dialog-1: [] (истории нет)
rewrite без истории возвращает вопрос как есть: Как его обучали?


## Выводы

1. Сокращённый индекс теряет все вопросы вне своего покрытия (Hit Rate@3: 50% против 100%), а LLM при этом галлюцинирует. Промпт отсутствующие в индексе сведения не вернёт: контекст собирается только из результатов поиска.
2. Размер фрагментов 500/1000/2000 дал одинаковые 70% — на этом документе разницы нет, различаются только провалившиеся вопросы (у мелких теряется контекст, у крупных размывается тема).
3. Перекрытие 0/100/200: 4/5, 4/5, 5/5. Вопрос про «alignment tax» нашёлся только с перекрытием 200. Дублей в top-3 не было.
4. Эмбеддинги сильнее мешка слов (80%/60% против 60%/0%). Мешок слов работает только при точных терминах; на русских запросах дважды дал нулевой вектор.
5. Ссылки корректны в 4 из 5 ответов (id, страница, подтверждение текстом); в одном ответе ссылку модель не вывела. Оформленная ссылка сама по себе достоверность не гарантирует.
6. Строгий промпт: 5/5 корректных отказов и ни одного выдуманного ответа (исходный: 4/5 и одна выдумка про петафлопсы), средний балл не просел (2.0 против 1.8).
7. Переформулировка уточнений (вариант Б) нашла нужный фрагмент в 4 из 6 случаев; поиск по сырой реплике (А) — в 0 из 6. Типичная ошибка Б — лишние детали в переформулировке.

Итоговая конфигурация: chunk_size=1000, overlap=200, k=3, эмбеддинги bge-m3, qwen2.5:3b (температура 0), строгий промпт со ссылками на источники, память диалога с переформулировкой уточнений.